In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check the GPU status
!nvidia-smi

Fri Sep 25 16:08:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip install evaluate

In [4]:
# Fix ImportError: cannot import name 'is_offline_mode' from 'huggingface_hub'
# !uv remove huggingface_hub
# !uv pip install --upgrade huggingface_hub

# Fix ImportError: huggingface-hub>=1.5.0,<2.0 is required for a normal functioning of this module, 
# but found huggingface-hub==2.0.0.
!uv remove huggingface_hub
!uv pip install huggingface-hub>=1.5.0,<2.0

error: The dependency `huggingface-hub` could not be found in `project.dependencies`
/bin/bash: line 1: 2.0: No such file or directory


In [5]:
from datetime import datetime

import json
import evaluate
import pandas as pd
from transformers import (
    AutoTokenizer, 
    AutoModelForQuestionAnswering, 
    DataCollatorWithPadding, 
    Trainer, 
    TrainingArguments,
)
from datasets import load_dataset

# Configurations

In [6]:
# Run configuration
SEED = 42
LANGUAGES = ['en', 'ar', 'de', 'el', 'es', 'hi', 'ru', 'th', 'tr', 'vi', 'zh']

# Model configuration
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LoRA-mrg-v260922221946'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-15K-s42-LoRA-mrg-v260925013329'
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-add-v260925155115'
MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333'

# Data configuration
DATA_ID = 'google/xquad'
DATA_DIR = 'xquad.{lang}'
DATA_SPLIT = 'validation'
# TEST_SIZE = 125
# TEST_SIZE = 625
TEST_SIZE = -1  # -1 = all samples

# Evaluation configuration
BATCH_SIZE = 16

# Set up the evaluation directory
eval_dir = f'./eval/xquad_xlmr/{TEST_SIZE}/{MODEL_ID}'
print(f"Evaluation directory: {eval_dir}")

Evaluation directory: ./eval/xquad_xlmr/-1/alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333


# Utilities

In [7]:
def load_test_dataset(
    lang, # e.g., 'en' | 'ja' | 'id'
    size,
    data_id=DATA_ID,
    data_dir=DATA_DIR,
    data_split=DATA_SPLIT,
):
    assert '{lang}' in data_dir, "Data directory must contain a '{lang}' placeholder."
    
    # Load the full split (non-streaming) so the total count is known up front.
    dataset = load_dataset(
        data_id,
        data_dir=data_dir.format(lang=lang),
        split=data_split,
    )

    # A negative size (e.g., -1) or None means "use all samples".
    if size is not None and 0 < size < len(dataset):
        dataset = dataset.select(range(size))

    if len(dataset) == 0:
        raise ValueError(f"Loaded 0 examples for lang={lang!r} with size={size!r}.")

    print(f"Loaded {len(dataset)} examples for lang={lang!r} (requested size={size}).")

    return dataset

# Model

In [8]:
# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID, device_map='auto')
model.eval()

print("device:", model.device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

device: cuda:0


# Data

In [9]:
# Evaluation-only preprocessing.
#
# Deliberately different from the training preprocessing:
#   - sliding windows (stride) so no part of the context is ever cut off, but
#     EVERY window is kept -- ground truth must never decide which windows the
#     model is scored on (that would leak the answer);
#   - no start/end positions -- eval compares predicted strings against
#     reference strings (EM/F1), so labels are neither needed nor wanted;
#   - overflow_to_sample_mapping is kept as `example_id` so that each feature
#     can be grouped back into its parent example before scoring.
def preprocess_squad_eval(examples):
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second', # Only truncate the context
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=False,
    )

    # Each generated window points back to its original xQuAD example.
    sample_mapping = tokenized.pop('overflow_to_sample_mapping')

    # One id per WINDOW (several windows can share the same example id).
    tokenized['example_id'] = [examples['id'][i] for i in sample_mapping]

    # Which tokens belong to the context:
    # 0 = question / special token, 1 = context.
    # Needed later to keep predicted spans inside the context.
    tokenized['context_mask'] = [
        [1 if seq_id == 1 else 0 for seq_id in tokenized.sequence_ids(i)]
        for i in range(len(tokenized['input_ids']))
    ]

    return tokenized

In [10]:
def postprocess_qa_predictions(
    raw_dataset,       # ground-truth examples (id / context / answers)
    example_ids,       # one id per feature (window)
    context_masks,     # one context mask per feature
    offset_mappings,   # one offset mapping per feature
    start_logits,      # (n_features, seq_len)
    end_logits,        # (n_features, seq_len)
    n_best=20,
    max_answer_length=30,
):
    """Group windows back into their parent example and pick ONE span each.

    Every window nominates its best candidate span; candidates from all
    windows of the SAME example then compete on score, and the winner becomes
    that example's single prediction_text. Ground truth is touched only here,
    at scoring time -- never while selecting candidates.
    """
    id_to_context = {ex['id']: ex['context'] for ex in raw_dataset}

    # 1) Collect candidate spans, grouped per example.
    candidates = {}  # example_id -> [(score, start_tok, end_tok, text), ...]

    for feat_idx, (s_log, e_log) in enumerate(zip(start_logits, end_logits)):
        ex_id = example_ids[feat_idx]
        context = id_to_context[ex_id]
        offsets = offset_mappings[feat_idx]
        mask = context_masks[feat_idx]

        context_tokens = [tok for tok, flag in enumerate(mask) if flag]
        if not context_tokens:
            continue

        # Only the top-k starts/ends can ever win, so skip the full O(n^2) scan.
        k = min(n_best * 2, len(context_tokens))
        top_starts = sorted(context_tokens, key=lambda t: s_log[t], reverse=True)[:k]
        top_ends = sorted(context_tokens, key=lambda t: e_log[t], reverse=True)[:k]

        bucket = candidates.setdefault(ex_id, [])
        for start_tok in top_starts:
            start_char = offsets[start_tok][0]
            for end_tok in top_ends:
                if end_tok < start_tok:
                    continue
                if end_tok - start_tok + 1 > max_answer_length:
                    continue
                end_char = offsets[end_tok][1]
                if end_char <= start_char:
                    continue
                score = float(s_log[start_tok] + e_log[end_tok])
                text = context[start_char:end_char]
                bucket.append((score, start_tok, end_tok, text))

    # 2) Exactly one prediction per example.
    predictions = []
    for ex in raw_dataset:
        ex_id = ex['id']
        cands = candidates.get(ex_id)
        if not cands:
            predictions.append({'id': ex_id, 'prediction_text': ''})
            continue
        cands.sort(key=lambda c: c[0], reverse=True)

        # Highest-scoring span, skipping duplicate strings.
        best_text = None
        seen = set()
        for cand in cands:
            if cand[3] in seen:
                continue
            seen.add(cand[3])
            best_text = cand[3]
            break

        predictions.append({'id': ex_id, 'prediction_text': best_text or ''})

    # 3) References -- scoring is string-based (EM/F1), answer_start is unused.
    references = [
        {
            'id': ex['id'],
            'answers': {
                'text': ex['answers']['text'],
                'answer_start': ex['answers']['answer_start'],
            },
        }
        for ex in raw_dataset
    ]

    metrics = squad_metric.compute(predictions=predictions, references=references)
    return metrics, predictions

# Evaluation

In [11]:
# Set up a trainer for evaluation
data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
eval_args = TrainingArguments(
    output_dir=eval_dir,
    do_train=False,
    do_eval=True,
    per_device_eval_batch_size=BATCH_SIZE,
    dataloader_drop_last=False,
    report_to=[],
)
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=eval_args,
)
trainer.label_names = ['start_positions', 'end_positions']

In [12]:
results = {}
squad_metric = evaluate.load('squad')

# We score per EXAMPLE ourselves; the Trainer must not score per feature.
trainer.compute_metrics = None

for i, lang in enumerate(LANGUAGES):
    print(f"\n{'=' * 64}")
    print(f"[{i+1}/{len(LANGUAGES)}] Evaluating: {lang}")
    print(f"{'=' * 64}")

    # Load and preprocess the test dataset for this language
    raw_dataset = load_test_dataset(lang, size=TEST_SIZE)
    test_dataset = raw_dataset.map(
        preprocess_squad_eval,
        batched=True,
        remove_columns=raw_dataset.column_names,
    )

    # Pull the bookkeeping columns OUT of the Dataset: they are not model
    # inputs, so the Trainer would silently strip them as unused columns.
    example_ids = test_dataset['example_id']
    context_masks = test_dataset['context_mask']
    offset_mappings = test_dataset['offset_mapping']
    test_dataset = test_dataset.remove_columns(
        ['example_id', 'context_mask', 'offset_mapping']
    )

    print(f"  {lang}: {len(raw_dataset)} raw examples -> {len(test_dataset)} features")

    # Run prediction (no labels => no loss, nothing that could leak)
    preds = trainer.predict(test_dataset)
    start_logits, end_logits = preds.predictions

    # Group windows back into examples and score once per example
    eval_metrics, pred_texts = postprocess_qa_predictions(
        raw_dataset=raw_dataset,
        example_ids=example_ids,
        context_masks=context_masks,
        offset_mappings=offset_mappings,
        start_logits=start_logits,
        end_logits=end_logits,
    )

    results[lang] = {
        **preds.metrics,  # test_runtime, test_samples_per_second, ...
        'test_exact_match': eval_metrics['exact_match'],
        'test_f1': eval_metrics['f1'],
    }

    print(f"Exact Match: {results[lang]['test_exact_match']:.2f}%")
    print(f"F1: {results[lang]['test_f1']:.2f}%")

    # Save per-language predictions (one block per EXAMPLE, not per window)
    pred_path = f"{eval_dir}/predictions_{lang}.txt"
    with open(pred_path, 'w', encoding='utf-8') as f:
        for pred, ex in zip(pred_texts, raw_dataset):
            gt_text = ex['answers']['text'][0]
            pred_text = pred['prediction_text']

            f.write(f"Q: {ex['question']}\n")
            f.write(f"GT: {gt_text}\n")
            f.write(f"Pred: {pred_text}\n")
            # NOTE: quick eyeball check only -- official EM normalizes text,
            # so this "Match" line is not the same number as test_exact_match.
            f.write(f"Match: {pred_text.strip().lower() == gt_text.strip().lower()}\n")
            f.write("-" * 64 + "\n")

    print("-" * 64)
    print(f"Saved predictions to: {pred_path}")

# Print an overview table
print(f"\n{'=' * 64}")
print("Evaluation Result Overview")
print(f"{'=' * 64}")
print(f"{'Language':<10} {'Exact Match':>12} {'F1':>12}")
print("-" * 36)
for lang in LANGUAGES:
    print(f"{lang:<10} {results[lang]['test_exact_match']:>11.2f}% {results[lang]['test_f1']:>11.2f}%")

# Calculate averages
avg_em = sum(r['test_exact_match'] for r in results.values()) / len(results)
avg_f1 = sum(r['test_f1'] for r in results.values()) / len(results)

print("-" * 36)
print(f"{'Average':<10} {avg_em:>11.2f}% {avg_f1:>11.2f}%")


[1/11] Evaluating: en
Loaded 1190 examples for lang='en' (requested size=-1).


Map:   0%|          | 0/1190 [00:00<?, ? examples/s]

  en: 1190 raw examples -> 1249 features


Exact Match: 33.53%
F1: 43.62%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/-1/alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333/predictions_en.txt

[2/11] Evaluating: ar
Loaded 1190 examples for lang='ar' (requested size=-1).


Map:   0%|          | 0/1190 [00:00<?, ? examples/s]

...

  vi: 1190 raw examples -> 1287 features


Exact Match: 20.84%
F1: 32.46%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/-1/alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333/predictions_vi.txt

[11/11] Evaluating: zh
Loaded 1190 examples for lang='zh' (requested size=-1).


Map:   0%|          | 0/1190 [00:00<?, ? examples/s]

  zh: 1190 raw examples -> 1230 features


Exact Match: 17.14%
F1: 23.58%
----------------------------------------------------------------
Saved predictions to: ./eval/xquad_xlmr/-1/alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333/predictions_zh.txt

Evaluation Result Overview
Language    Exact Match           F1
------------------------------------
en               33.53%       43.62%
ar               18.66%       27.99%
de               25.55%       33.67%
el               23.28%       32.82%
es               25.21%       34.81%
hi               22.52%       31.22%
ru               21.76%       32.14%
th               21.09%       27.39%
tr               17.23%       26.00%
vi               20.84%       32.46%
zh               17.14%       23.58%
------------------------------------
Average          22.44%       31.43%


In [13]:
# Save metrics JSON and CSV
metrics_json_path = f'{eval_dir}/metrics.json'
metrics_csv_path = f'{eval_dir}/metrics.csv'

metrics = []
for lang in LANGUAGES:
    metrics.append({
        'lang': lang,
        **results[lang],
    })
metrics_df = pd.DataFrame(metrics)
# test_loss is intentionally absent: eval uses no labels (EM/F1 only).
metrics_df = metrics_df[[
    'lang', 'test_exact_match', 'test_f1',
    'test_model_preparation_time', 'test_runtime',
    'test_samples_per_second', 'test_steps_per_second'
]] # Rearrange columns

metrics_df.to_json(metrics_json_path, orient='records')
metrics_df.to_csv(metrics_csv_path, index=False)

print(f"Saved metrics JSON to: {metrics_json_path}")
print(f"Saved metrics CSV to: {metrics_csv_path}")

Saved metrics JSON to: ./eval/xquad_xlmr/-1/alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333/metrics.json
Saved metrics CSV to: ./eval/xquad_xlmr/-1/alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333/metrics.csv


In [14]:
# Save metadata JSON
metadata_json_path = f'{eval_dir}/metadata.json'
metadata = {
    'model_id': MODEL_ID,
    'data_id': DATA_ID,
    'data_dir': DATA_DIR,
    'data_split': DATA_SPLIT,
    'test_size': TEST_SIZE,
    'batch_size': BATCH_SIZE,
    'evaluated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}

with open(metadata_json_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=4)
    
print(f"Saved metadata JSON to: {metadata_json_path}")

Saved metadata JSON to: ./eval/xquad_xlmr/-1/alxxtexxr/XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333/metadata.json


In [15]:
# Create a zip file containing the evaluation results
import shutil

# zip_dir = str(Path(eval_dir).parent.parent)
# zip_name = zip_dir.replace('/', '_').replace('\\', '_')
zip_dir = eval_dir
zip_name = eval_dir.rsplit('/', 1)[-1]

shutil.make_archive(zip_name, 'zip', zip_dir)

print(f"Created a zip file: {zip_name}.zip")

Created a zip file: XLM-R-Base-squad-el-s42-LoRA-avg-v260925155333.zip
